# Notebook 01 — LLM Relevance Judges vs. Human Judges (MIRACL, Indian languages)

**Kaggle setup (right-hand panel) before running:**
- Session options → Accelerator: **GPU T4 x2** (or GPU P100)
- Session options → Internet: **On**
- Add-ons → Secrets → add a secret named **`HF_TOKEN`** (Hugging Face read token) and attach it

Then click **Run All**. Results are saved after every batch; if the session times out, Run All again and it resumes.
Click **Save Version** at the end to keep the files in `/kaggle/working/results`.

## 0. Install packages

In [ ]:
import subprocess, sys
# Upgrade only the LLM libraries; keep Kaggle's pandas / scikit-learn versions
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas<3", "scikit-learn<1.9",
                "-U", "transformers", "accelerate", "bitsandbytes", "huggingface_hub"], check=True)
print("Install finished. If this is the first run in this session: Run -> Restart session, then Run All again.")

import os, gc, gzip, json, random, time
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"   # reduce GPU memory fragmentation
from collections import defaultdict

import pandas as pd
import torch
from tqdm.auto import tqdm
from huggingface_hub import list_repo_files, hf_hub_download, login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score, roc_auc_score

## 1. Configuration  (START SMALL: 20 queries for the first test run)

In [ ]:
IN_KAGGLE = os.path.exists("/kaggle")
CONFIG = {
    "languages": ["hi", "bn", "te"],
    "split": "dev",
    "queries_per_language": 100,         # full run (pairs judged earlier are reused)
    "models": [
        "Qwen/Qwen2.5-7B-Instruct",
        # Add more judges one at a time after the first run works:
        # "meta-llama/Llama-3.1-8B-Instruct",   # gated: accept terms on HF first
        # "CohereLabs/aya-expanse-8b",          # gated: accept terms on HF first
        # "google/gemma-2-9b-it",               # gated: accept terms on HF first
    ],
    "prompt_languages": ["english"],     # later: ["english", "native"]
    "batch_size": 4,                     # halves automatically if a batch runs out of memory
    "max_passage_chars": 8000,           # effectively no character cut
    "max_passage_tokens": 2048,          # 512 cut 40-73% of passages; 2048 keeps almost all
    "require_both_labels": True,         # keep only queries with >=1 relevant AND >=1 non-relevant passage
    "seed": 42,
    "output_dir": "/kaggle/working/results" if IN_KAGGLE else "results",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU - turn it on!")

## 2. Hugging Face login

In [ ]:
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
    print("Logged in to Hugging Face")
else:
    print("WARNING: no HF_TOKEN secret found - gated models/datasets will fail")

## 3. Load MIRACL topics, human qrels, and only the judged passages

In [ ]:
TOPICS_REPO = "miracl/miracl"
CORPUS_REPO = "miracl/miracl-corpus"
topic_files = list_repo_files(TOPICS_REPO, repo_type="dataset")
corpus_files = list_repo_files(CORPUS_REPO, repo_type="dataset")


def find_one(files, basename, lang):
    hits = [f for f in files if f.endswith(basename)]
    if not hits:
        related = [f for f in files if f"-{lang}/" in f or f"-{lang}-" in f][:30]
        raise FileNotFoundError(
            f"'{basename}' not found. Send this list to fix the path:\n" + "\n".join(related))
    return hits[0]


def load_topics(lang, split):
    fname = find_one(topic_files, f"topics.miracl-v1.0-{lang}-{split}.tsv", lang)
    path = hf_hub_download(TOPICS_REPO, fname, repo_type="dataset")
    topics = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                topics[parts[0]] = parts[1]
    return topics


def load_qrels(lang, split):
    fname = find_one(topic_files, f"qrels.miracl-v1.0-{lang}-{split}.tsv", lang)
    path = hf_hub_download(TOPICS_REPO, fname, repo_type="dataset")
    qrels = defaultdict(dict)
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            if len(parts) == 4:
                qid, _, docid, rel = parts
                qrels[qid][docid] = int(rel)
    return qrels


def load_passages(lang, needed_docids):
    shards = sorted(f for f in corpus_files
                    if f"miracl-corpus-v1.0-{lang}/" in f and f.endswith(".jsonl.gz"))
    if not shards:
        raise FileNotFoundError(f"No corpus shards for '{lang}'. Example files: {corpus_files[:20]}")
    needed, found = set(needed_docids), {}
    for shard in shards:
        path = hf_hub_download(CORPUS_REPO, shard, repo_type="dataset")
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                if doc["docid"] in needed:
                    found[doc["docid"]] = (doc.get("title", ""), doc["text"])
        print(f"  {shard}: {len(found)}/{len(needed)} passages found")
        if len(found) == len(needed):
            break
    return found


def build_pairs(lang):
    rng = random.Random(CONFIG["seed"])
    topics = load_topics(lang, CONFIG["split"])
    qrels = load_qrels(lang, CONFIG["split"])
    qids = sorted(q for q in qrels if q in topics)
    if CONFIG["require_both_labels"]:
        total = len(qids)
        qids = [q for q in qids if 0 < sum(qrels[q].values()) < len(qrels[q])]
        print(f"{lang}: {len(qids)}/{total} queries have both relevant and non-relevant judgments")
    rng.shuffle(qids)
    qids = qids[: CONFIG["queries_per_language"]]
    passages = load_passages(lang, {d for q in qids for d in qrels[q]})
    pairs = []
    for q in qids:
        for d, rel in qrels[q].items():
            if d in passages:
                title, text = passages[d]
                pairs.append({"lang": lang, "qid": q, "docid": d, "query": topics[q],
                              "title": title, "text": text[: CONFIG["max_passage_chars"]],
                              "human": rel})
    pos = sum(p["human"] for p in pairs)
    print(f"{lang}: {len(qids)} queries, {len(pairs)} pairs "
          f"({pos} relevant, {len(pairs) - pos} non-relevant)")
    return pairs


print("\n=== Loading MIRACL data ===")
DATA = {lang: build_pairs(lang) for lang in CONFIG["languages"]}

## 4. Prompts  (native Bengali/Telugu: have a native speaker write them)

In [ ]:
PROMPTS = {
    "english": {
        "system": "You are a careful search quality rater.",
        "user": ("Judge whether the passage is relevant to the query. A passage is relevant if it "
                 "contains information that helps answer the query.\n\n"
                 "Query: {query}\n\nPassage title: {title}\nPassage: {text}\n\n"
                 "Answer with a single digit: 1 if relevant, 0 if not relevant."),
    },
    "native": {
        "hi": {
            "system": "आप एक सावधान खोज गुणवत्ता मूल्यांकनकर्ता हैं।",
            "user": ("तय कीजिए कि दिया गया अनुच्छेद प्रश्न के लिए प्रासंगिक है या नहीं। अनुच्छेद प्रासंगिक है "
                     "यदि उसमें ऐसी जानकारी है जो प्रश्न का उत्तर देने में मदद करती है।\n\n"
                     "प्रश्न: {query}\n\nअनुच्छेद का शीर्षक: {title}\nअनुच्छेद: {text}\n\n"
                     "केवल एक अंक में उत्तर दीजिए: प्रासंगिक हो तो 1, प्रासंगिक न हो तो 0।"),
        },
        # "bn": {"system": "...", "user": "..."},   # TODO native speaker
        # "te": {"system": "...", "user": "..."},   # TODO native speaker
    },
}


def get_template(lang, prompt_language):
    if prompt_language == "native":
        if lang in PROMPTS["native"]:
            return PROMPTS["native"][lang]
        return None          # skip: no native prompt yet
    return PROMPTS["english"]

## 5. Judge model helpers

In [ ]:
def load_judge(name):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"                     # last position = real last token
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16),
        device_map="auto",
    )
    mdl.eval()

    def one_token(s):
        ids = tok.encode(s, add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f"'{s}' is {len(ids)} tokens for {name}: {ids}")
        return ids[0]

    return tok, mdl, one_token("0"), one_token("1")


def truncate_tokens(tok, text, max_tokens):
    ids = tok.encode(text, add_special_tokens=False)
    return text if len(ids) <= max_tokens else tok.decode(ids[:max_tokens])


def build_prompt(tok, p, tpl):
    text = truncate_tokens(tok, p["text"], CONFIG["max_passage_tokens"])
    user = tpl["user"].format(query=p["query"], title=p["title"], text=text)
    msgs = [{"role": "system", "content": tpl["system"]}, {"role": "user", "content": user}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:  # models without a system role (e.g. Gemma)
        msgs = [{"role": "user", "content": tpl["system"] + "\n\n" + user}]
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def judge_batch(tok, mdl, id0, id1, prompts):
    """Judge a batch; on out-of-memory, split it in half and retry."""
    try:
        return _judge(tok, mdl, id0, id1, prompts)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        mid = len(prompts) // 2
        return (judge_batch(tok, mdl, id0, id1, prompts[:mid]) +
                judge_batch(tok, mdl, id0, id1, prompts[mid:]))


@torch.no_grad()
def _judge(tok, mdl, id0, id1, prompts):
    enc = tok(prompts, return_tensors="pt", padding=True).to(mdl.device)
    try:
        out = mdl(**enc, logits_to_keep=1)        # newer transformers: last position only
    except TypeError:
        out = mdl(**enc)
    last = out.logits[:, -1, :].float()
    two = torch.stack([last[:, id0], last[:, id1]], dim=-1)
    probs = torch.softmax(two, dim=-1)[:, 1].cpu().tolist()   # P(relevant)
    logodds = (two[:, 1] - two[:, 0]).cpu().tolist()          # exact score, no saturation near 1
    return list(zip(probs, logodds))


def results_path(lang, model_name, prompt_language):
    short = model_name.split("/")[-1]
    tag = f"t{CONFIG['max_passage_tokens']}"
    return os.path.join(CONFIG["output_dir"], f"{lang}__{short}__{prompt_language}__{tag}.jsonl")

## 6. Run all judgments (resumable: re-run to continue after a timeout)

In [ ]:
for model_name in CONFIG["models"]:
    print(f"\n=== Loading judge: {model_name} ===")
    tok, mdl, ID0, ID1 = load_judge(model_name)
    print("Label token ids:", ID0, ID1)

    for prompt_language in CONFIG["prompt_languages"]:
        for lang, pairs in DATA.items():
            tpl = get_template(lang, prompt_language)
            if tpl is None:
                print(f"Skipping {lang}/{prompt_language}: no native prompt yet")
                continue
            n_tok = [len(tok.encode(p["text"], add_special_tokens=False)) for p in pairs]
            tok_len = {(p["qid"], p["docid"]): n for p, n in zip(pairs, n_tok)}
            if prompt_language == CONFIG["prompt_languages"][0]:
                n_trunc = sum(n > CONFIG["max_passage_tokens"] for n in n_tok)
                stats = {"lang": lang, "model": model_name, "pairs": len(pairs), "truncated": n_trunc,
                         "truncated_pct": round(100 * n_trunc / max(len(pairs), 1), 1),
                         "median_tokens": sorted(n_tok)[len(n_tok) // 2] if n_tok else 0}
                print(f"  truncation {lang}: {n_trunc}/{len(pairs)} passages over "
                      f"{CONFIG['max_passage_tokens']} tokens ({stats['truncated_pct']}%), median {stats['median_tokens']} tokens")
                with open(os.path.join(CONFIG["output_dir"], "truncation_stats.jsonl"), "a", encoding="utf-8") as sf:
                    sf.write(json.dumps(stats) + "\n")
            path = results_path(lang, model_name, prompt_language)
            done = set()
            if os.path.exists(path):
                with open(path, encoding="utf-8") as f:
                    done = {(r["qid"], r["docid"]) for r in map(json.loads, f)}
            todo = [p for p in pairs if (p["qid"], p["docid"]) not in done]
            todo.sort(key=lambda p: len(p["text"]))      # similar lengths per batch = less padding
            print(f"{lang}/{prompt_language}: {len(done)} done, {len(todo)} to go")
            bs, t0 = CONFIG["batch_size"], time.time()
            with open(path, "a", encoding="utf-8") as f:
                for i in tqdm(range(0, len(todo), bs), desc=f"{lang}-{prompt_language}"):
                    batch = todo[i:i + bs]
                    probs = judge_batch(tok, mdl, ID0, ID1, [build_prompt(tok, p, tpl) for p in batch])
                    for p, (pr, lo) in zip(batch, probs):
                        f.write(json.dumps({
                            "lang": lang, "qid": p["qid"], "docid": p["docid"],
                            "human": p["human"], "p_relevant": pr, "logodds": lo,
                            "passage_tokens": tok_len[(p["qid"], p["docid"])],
                            "passage_chars": len(p["text"]),
                            "model": model_name, "prompt_language": prompt_language,
                        }, ensure_ascii=False) + "\n")
                    f.flush()
            if todo:
                secs = time.time() - t0
                print(f"  {len(todo)} pairs in {secs:.0f}s  ({len(todo) / secs:.2f} pairs/sec)")

    del mdl, tok                       # free GPU before the next judge
    gc.collect()
    torch.cuda.empty_cache()

## 7. Agreement with human judges

In [ ]:
rows = []
for model_name in CONFIG["models"]:
    for prompt_language in CONFIG["prompt_languages"]:
        for lang in CONFIG["languages"]:
            path = results_path(lang, model_name, prompt_language)
            if not os.path.exists(path):
                continue
            df = pd.read_json(path, lines=True, dtype={"qid": str, "docid": str})
            keep = {(p["qid"], p["docid"]) for p in DATA[lang]}      # only pairs in the current sample
            df = df[[(q, d) in keep for q, d in zip(df["qid"], df["docid"])]]
            df = df.drop_duplicates(["qid", "docid"])
            cols = [c for c in ["qid", "docid", "human", "p_relevant", "logodds", "passage_tokens", "passage_chars"] if c in df.columns]
            df[cols].to_csv(
                path.replace(".jsonl", "__clean.csv"), index=False)
            pred = (df["p_relevant"] >= 0.5).astype(int)
            rows.append({
                "model": model_name.split("/")[-1], "prompt": prompt_language, "lang": lang,
                "pairs": len(df),
                "kappa": cohen_kappa_score(df["human"], pred),
                "accuracy": accuracy_score(df["human"], pred),
                "f1": f1_score(df["human"], pred, zero_division=0),
                "auc": roc_auc_score(df["human"], df["p_relevant"])
                       if df["human"].nunique() > 1 else float("nan"),
                "human_pos_rate": df["human"].mean(),
                "llm_pos_rate": pred.mean(),
                "overrating_gap": pred.mean() - df["human"].mean(),
            })

summary = pd.DataFrame(rows).round(3)
summary.to_csv(os.path.join(CONFIG["output_dir"], "summary.csv"), index=False)
print("\n=== RESULTS ===")
print(summary.to_string(index=False))
print(f"\nSaved to {CONFIG['output_dir']}  (click 'Save Version' to keep the files)")